In [ ]:


from google.colab import drive
drive.mount('/content/drive')

import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


SAVE_DIR = (
    "/content/drive/MyDrive/MS_Research/"
    "results/adaptive_pdd_full50k"
)

os.makedirs(SAVE_DIR, exist_ok=True)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

train_dataset = datasets.CIFAR10(
    root="/content/data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="/content/data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

model = mobilenet_v2(weights=None)

model.classifier[1] = nn.Linear(
    model.last_channel,
    10
)

model = model.to(device)

print("\nTraining samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Batches per epoch:", len(train_loader))
print("Model: MobileNet-V2")

In [ ]:

EPOCHS = 20

# CONVERGENCE  CONDITIONS

WINDOW = 3

# Average training accuracy change < 0.5 percentage points
ACC_THRESHOLD = 0.005

# Average relative training loss change < 3%
LOSS_THRESHOLD = 0.03

# Condition must hold for 2 consecutive checks
PATIENCE = 2

# Do not allow dropout too early
MIN_WARMUP = 5

ALPHA = 0.05
MAX_DROP_RATE = 0.50

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=1,
    gamma=0.98
)

# RESULT STORAGE

history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
    "drop_rate": [],
    "samples_used": [],
    "epoch_time": [],
    "mean_acc_change": [],
    "mean_loss_change": []
}

dropout_started = False
dropout_start_epoch = None
stable_counter = 0

# TEST FUNCTION

def evaluate_test():

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return (
        running_loss / total,
        correct / total
    )


# TRAINING

print("=" * 70)
print("ADAPTIVE-START PROGRESSIVE DATA DROPOUT")
print("CIFAR-10: FULL 50,000 TRAINING SAMPLES")
print("=" * 70)

for epoch in range(EPOCHS):

    epoch_start = time.time()

    if dropout_started:

        epochs_after_start = (
            epoch - dropout_start_epoch + 1
        )

        drop_rate = min(
            MAX_DROP_RATE,
            ALPHA * epochs_after_start
        )

    else:

        drop_rate = 0.0


    model.train()

    running_loss = 0.0
    correct = 0
    samples_used = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        if drop_rate > 0:

            model.eval()

            with torch.no_grad():

                check_outputs = model(images)

                probabilities = torch.softmax(
                    check_outputs,
                    dim=1
                )

                true_class_probability = probabilities[
                    torch.arange(
                        labels.size(0),
                        device=device
                    ),
                    labels
                ]


            keep_count = max(
                1,
                int(
                    round(
                        labels.size(0)
                        * (1.0 - drop_rate)
                    )
                )
            )

            keep_indices = torch.argsort(
                true_class_probability
            )[:keep_count]

            images = images[keep_indices]
            labels = labels[keep_indices]

        # TRAIN SELECTED DATA

        model.train()

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()


        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        samples_used += labels.size(0)


    scheduler.step()

    # TRAINING METRICS

    train_loss = (
        running_loss / samples_used
    )

    train_accuracy = (
        correct / samples_used
    )



    test_loss, test_accuracy = evaluate_test()

    epoch_time = time.time() - epoch_start


    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["test_loss"].append(
        test_loss
    )

    history["test_accuracy"].append(
        test_accuracy
    )

    history["drop_rate"].append(
        drop_rate
    )

    history["samples_used"].append(
        samples_used
    )

    history["epoch_time"].append(
        epoch_time
    )

    # CONVERGENCE DETECTION


    mean_acc_change = None
    mean_loss_change = None


    if (
        not dropout_started
        and
        epoch + 1 >= MIN_WARMUP
        and
        len(history["train_accuracy"]) >= WINDOW + 1
    ):

        acc_changes = []
        loss_changes = []


        for i in range(
            len(history["train_accuracy"]) - WINDOW,
            len(history["train_accuracy"])
        ):


            # Accuracy change

            acc_change = abs(
                history["train_accuracy"][i]
                -
                history["train_accuracy"][i - 1]
            )

            acc_changes.append(
                acc_change
            )

            # Relative loss change

            previous_loss = (
                history["train_loss"][i - 1]
            )

            loss_change = abs(
                history["train_loss"][i]
                -
                previous_loss
            ) / (
                abs(previous_loss) + 1e-8
            )

            loss_changes.append(
                loss_change
            )


        mean_acc_change = np.mean(
            acc_changes
        )

        mean_loss_change = np.mean(
            loss_changes
        )



        # Check stability


        if mean_acc_change < ACC_THRESHOLD:
            stable_counter += 1
        else:
            stable_counter = 0



        # Start adaptive PDD


        if stable_counter >= PATIENCE:

            dropout_started = True

            # Dropout starts NEXT epoch
            dropout_start_epoch = epoch + 1


    history["mean_acc_change"].append(
        mean_acc_change
    )

    history["mean_loss_change"].append(
        mean_loss_change
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n" + "-" * 70)

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"Train Accuracy: "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Train Loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Test Accuracy: "
        f"{test_accuracy * 100:.2f}%"
    )

    print(
        f"Drop Rate: "
        f"{drop_rate * 100:.1f}%"
    )

    print(
        f"Samples Used: "
        f"{samples_used}/{len(train_dataset)}"
    )

    print(
        f"Epoch Time: "
        f"{epoch_time:.1f}s"
    )


    if mean_acc_change is not None:

        print(
            f"Mean Accuracy Change: "
            f"{mean_acc_change:.6f}"
        )

        print(
            f"Mean Loss Change: "
            f"{mean_loss_change:.6f}"
        )

        print(
            f"Stable Counter: "
            f"{stable_counter}/{PATIENCE}"
        )


    if (
        dropout_start_epoch is not None
        and
        dropout_start_epoch == epoch + 1
    ):

        print()
        print(
            ">>> CONVERGENCE DETECTED <<<"
        )

        print(
            "Adaptive dropout begins "
            "next epoch."
        )


    # SAVE CHECKPOINT


    torch.save(
        {
            "epoch": epoch + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "history":
                history,

            "dropout_started":
                dropout_started,

            "dropout_start_epoch":
                dropout_start_epoch,

            "stable_counter":
                stable_counter
        },

        os.path.join(
            SAVE_DIR,
            "adaptive_pdd_checkpoint.pt"
        )
    )


# SAVE FINAL MODEL


torch.save(
    model.state_dict(),
    os.path.join(
        SAVE_DIR,
        "adaptive_pdd_final.pt"
    )
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

ADAPTIVE-START PROGRESSIVE DATA DROPOUT
CIFAR-10: FULL 50,000 TRAINING SAMPLES


In [ ]:

final_test_loss, final_test_accuracy = evaluate_test()

total_samples_used = sum(
    history["samples_used"]
)

full_data_samples = (
    len(train_dataset) * EPOCHS
)

sample_reduction = (
    1 -
    total_samples_used / full_data_samples
) * 100

total_time = sum(
    history["epoch_time"]
)


print("=" * 65)
print("FINAL ADAPTIVE PDD RESULTS")
print("=" * 65)

print(
    f"Test Accuracy: "
    f"{final_test_accuracy * 100:.2f}%"
)

print(
    f"Test Loss: "
    f"{final_test_loss:.4f}"
)

if dropout_start_epoch is not None:

    print(
        "Dropout Start Epoch:",
        dropout_start_epoch + 1
    )

else:

    print(
        "Dropout Start Epoch: Not triggered"
    )

print(
    f"Total Samples Used: "
    f"{total_samples_used:,}"
)

print(
    f"Full-data Samples: "
    f"{full_data_samples:,}"
)

print(
    f"Sample Reduction: "
    f"{sample_reduction:.2f}%"
)

print(
    f"Total Training Time: "
    f"{total_time / 60:.2f} minutes"
)

#Plot

epochs = range(
    1,
    EPOCHS + 1
)

plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    [
        acc * 100
        for acc in history["test_accuracy"]
    ],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.title("Adaptive PDD Accuracy")
plt.grid(True)
plt.show()


# DROPOUT GRAPH


plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    [
        rate * 100
        for rate in history["drop_rate"]
    ],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Data Dropped (%)")
plt.title("Adaptive PDD Drop Rate")
plt.grid(True)
plt.show()